# Parte 1 — Detecção por quadro e identidade ao longo do tempo

Este notebook usa os módulos do repositório para comparar o baseline IoU nas duas fontes exigidas: FRCNN público e Faster R-CNN ResNet-50 FPN pré-treinado no COCO, classe `person`.

**Objetivo:** medir quando a qualidade espacial da detecção não se traduz em identidade consistente. Não há treinamento nesta parte. As caixas públicas ficam congeladas para as Partes 2–5.

Instale `requirements-detector.txt` no Python usado pelo kernel. Em máquina com RTX 4050, instale o PyTorch com suporte a CUDA conforme o [seletor oficial](https://pytorch.org/get-started/locally/). Os mesmos módulos funcionam em CPU, GPU e Colab; no Colab basta clonar o repositório e executá-lo a partir de sua raiz.

As células de download e inferência pesada são opcionais e desligadas por padrão. A avaliação final só fica completa depois que ambas as fontes tiverem os três vídeos inteiros.

In [ ]:
from pathlib import Path
import os, sys, json, csv
from IPython.display import display, Image, Markdown

# Funciona abrindo o notebook da raiz ou de uma pasta interna do repositório.
REPO = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "configs/parte1.json").exists()), None)
if REPO is None:
    raise RuntimeError("Abra este notebook dentro do repositório clonado.")
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from mot_data import load_config, Sequence, valid_gt
from parte1 import evaluate_all
CONFIG = load_config(os.environ.get("PA2_CONFIG", "configs/parte1.json"))
DATA = Path(os.environ.get("PA2_DATA_ROOT", "data/MOT17"))
OUTPUT = Path(os.environ.get("PA2_OUTPUT", "outputs/parte1"))
CACHE = Path(os.environ.get("PA2_CACHE", "data/detections/torchvision"))
DEVICE = "auto"
RUN_DOWNLOAD = os.environ.get("PA2_RUN_DOWNLOAD", "0") == "1"
RUN_DETECT = os.environ.get("PA2_RUN_DETECT", "0") == "1"
RUN_BENCHMARK = os.environ.get("PA2_RUN_BENCHMARK", "0") == "1"
RUN_VISUALS = os.environ.get("PA2_RUN_VISUALS", "1") == "1"
print("Treino:", CONFIG["split"]["train"])
print("Validação:", CONFIG["split"]["validation"])
print("Fonte padrão:", CONFIG["public_detector"])
print("Inferência pesada habilitada:", RUN_DETECT)

## 1. Dados e split

O split é por vídeo físico: treino em 04, 05, 10 e 11; validação em 02, 09 e 13. Nenhuma cópia DPM/FRCNN/SDP cruza o split. A validação cobre densidades e pontos de vista distintos; é um conjunto de validação, não um teste independente para seleção contínua de modelos.

O pacote de labels basta para o baseline público. As imagens são necessárias ao detector torchvision e às figuras. O download seletivo usa HTTP Range e extrai somente FRCNN, evitando cópias duplicadas. Dados não são enviados ao Git.

MOT usa posições 1-based. `mot_data.py` converte para `xywh` 0-based internamente e preserva classe, marca de avaliação e visibilidade. Frames e IDs continuam 1-based.

In [ ]:
if RUN_DOWNLOAD:
    import subprocess
    subprocess.run([sys.executable, "download_mot17.py", "--root", str(DATA)], check=True)
    subprocess.run([sys.executable, "download_mot17.py", "--package", "frames", "--root", str(DATA),
                    "--sequences", *CONFIG["split"]["validation"]], check=True)

SEQUENCES = {name: Sequence.load(DATA, name, CONFIG["public_detector"])
             for name in CONFIG["split"]["train"] + CONFIG["split"]["validation"]}
for name, seq in SEQUENCES.items():
    gt = valid_gt(seq.ground_truth())
    print(f"{name}: {seq.length} quadros, {seq.width}x{seq.height}, "
          f"{seq.fps:g} FPS, densidade={len(gt)/seq.length:.1f}")

## 2. Inspeção das caixas

GT e detecções são apresentados separadamente para evitar confundir resposta correta com entrada do rastreador. As detecções **não têm identidade**. A visibilidade serve para análise; não é disponibilizada ao baseline.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image as PILImage
from visualization import draw_boxes

EXAMPLE = CONFIG["split"]["validation"][0]
seq = SEQUENCES[EXAMPLE]
if seq.image_path(1).exists():
    with PILImage.open(seq.image_path(1)) as image:
        pixels = np.asarray(image.convert("RGB"))
    gt = valid_gt(seq.ground_truth())
    gt = gt[gt[:, 0] == 1]
    det = seq.public_detections()
    det = det[(det[:, 0] == 1) & (det[:, 5] >= CONFIG["score_threshold"])]
    # IDs temporários servem apenas para desenhar caixas, não são IDs rastreados.
    boxes_for_display = np.column_stack([det[:, 0], np.arange(1, len(det)+1), det[:, 1:5]])
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, values, title in zip(axes, [gt, boxes_for_display], ["GT: IDs verdadeiros", "FRCNN público: caixas sem identidade"]):
        ax.imshow(pixels)
        draw_boxes(ax, values)
        ax.set_title(title)
        ax.axis("off")
    from io import BytesIO
    buffer = BytesIO()
    fig.savefig(buffer, format="png", dpi=120)
    display(Image(data=buffer.getvalue()))
    plt.close(fig)
else:
    display(Markdown("Imagens ainda ausentes; a avaliação pública abaixo funciona só com labels."))

## 3. Baseline e regras da avaliação

`baseline.py`, classe `IoUTracker`: Hungarian sobre IoU com limiar 0,3, nascimento imediato e tolerância de dois quadros intermediários sem observação. Usa a última caixa observada, sem movimento aprendido. Tracks ocultas não emitem caixas.

`metrics.py`: IDF1 com atribuição global um-para-um, switches comparados ao último ID associado, fragmentação como interrupção e retomada nas observações GT, erro absoluto de IDs únicos. IoU de avaliação: 0,5.

`mot_data.py`: pedestres com marca válida; visibilidade baixa ou zero não é excluída. Predições associadas a GT ignorado são desconsideradas **depois** da inferência, com prioridade a pedestres válidos. É nosso protocolo `pa2-pedestrians-v1`, não o avaliador oficial MOTChallenge.

`detection_metrics.py`: AP50 e mAP@0,50:0,95 com 101 pontos de recall. Matching espacial independente por imagem; agregação por sequência. A avaliação espacial não usa identidades. AP e tracking usam o mesmo corte de confiança 0,5. O AP é, portanto, limitado às detecções acima desse corte.

In [ ]:
# Fonte pública: avaliação completa das sete sequências, reutilizável pela Pessoa 2.
public_results = evaluate_all(DATA, CONFIG, OUTPUT, sources=("public",))
columns = ["sequence", "split", "ap50", "map_50_95", "idf1", "id_switches", "fragmentations", "predicted_identities", "gt_identities"]
table = "| " + " | ".join(columns) + " |\n| " + " | ".join(["---"] * len(columns)) + " |\n"
for row in public_results:
    table += "| " + " | ".join(f"{row[c]:.4f}" if isinstance(row[c], float) else str(row[c]) for c in columns) + " |\n"
display(Markdown(table))

## 4. Inferência torchvision e custo

Faster R-CNN ResNet-50 FPN, pesos COCO_V1, sem fine-tuning. O NMS interno do RPN e das caixas finais é substituído pelo nosso NMS em um escopo temporário, restaurado mesmo após erro.

Para executar a comparação, altere `RUN_BENCHMARK` e `RUN_DETECT` para `True` na primeira célula. `DEVICE="auto"` seleciona CUDA quando disponível. O benchmark usa cinco warmups e 30 quadros; mede tempo, memória e projeta custo para 1.875 quadros.

Cache: uma detecção por arquivo/quadro, incluindo quadros vazios; manifesto com hashes das imagens, detecções, código e configuração. A execução pode ser retomada. Mudanças incompatíveis geram erro, nunca reutilização silenciosa.

In [ ]:
if RUN_BENCHMARK or RUN_DETECT:
    from detector import build_detector, benchmark, extract_detections
    model, transform, device, versions = build_detector(CONFIG, DEVICE)
    OUTPUT.mkdir(parents=True, exist_ok=True)
    if RUN_BENCHMARK:
        print(benchmark(SEQUENCES[EXAMPLE], model, transform, device, OUTPUT / "benchmark.json"))
    if RUN_DETECT:
        for name in CONFIG["split"]["validation"]:
            extract_detections(SEQUENCES[name], CONFIG, CACHE, model, transform, device, versions)
    del model
else:
    display(Markdown("Inferência pesada desligada. Habilite-a na primeira célula ou execute `python parte1.py detect`."))

## 5. Comparação e gráfico obrigatório

O notebook tenta carregar a fonte torchvision nas três sequências **inteiras**. Cache ausente, parcial ou incompatível impede afirmar que a Parte 1 está completa.

Painel superior: mAP e IDF1. Painel inferior: razão de IDs e switches por identidade verdadeira. A ordem vem da densidade GT, não dos resultados. As escalas da razão de IDs e dos switches são indicadas nos dois eixos do painel inferior.

In [ ]:
from detector import load_cached_detections
cache_ready = True
try:
    for name in CONFIG["split"]["validation"]:
        load_cached_detections(SEQUENCES[name], CONFIG, CACHE)
except (FileNotFoundError, ValueError) as error:
    cache_ready = False
    display(Markdown(f"**Comparação torchvision pendente:** {error}"))

if cache_ready:
    results = evaluate_all(DATA, CONFIG, OUTPUT, sources=("public", "torchvision"), cache_dir=CACHE)
else:
    results = public_results
display(Image(filename=str(OUTPUT / "descolamento.png")))
payload = json.loads((OUTPUT / "results.json").read_text(encoding="utf-8"))
print("Parte 1 completa:", payload["complete_part1"])
print("Métricas salvas:", (OUTPUT / "metrics.csv").name)

## 6. Exemplos visuais e interpretação

As figuras mostram GT e baseline em linhas separadas, com cores determinísticas por identidade. Um ID previsto numericamente diferente do GT não significa erro; o que importa é sua consistência temporal. Os exemplos são escolhidos automaticamente por eventos observados, sem inventar falhas.

Investigue: a maior densidade sempre piorou IDF1? Movimento de câmera parece explicar exceções? Muitas identidades novas surgem mesmo quando o detector encontra caixas válidas? mAP e IDF1 medem propriedades diferentes e não precisam ter valores ou ordenação iguais.

In [ ]:
from visualization import example_strips
if RUN_VISUALS and all(SEQUENCES[EXAMPLE].image_path(f).exists() for f in range(1, SEQUENCES[EXAMPLE].length+1)):
    events = example_strips(SEQUENCES[EXAMPLE], OUTPUT / "predictions/public" / f"{EXAMPLE}.txt", OUTPUT / "examples", CONFIG)
    print(events)
    for path in sorted((OUTPUT / "examples").glob(f"{EXAMPLE}-*.png")):
        display(Image(filename=str(path)))
else:
    display(Markdown("Exemplos visuais desligados ou imagens incompletas."))

## 7. Contrato para as outras partes

- **Pessoa 2:** usar o split e o FRCNN público fixados em `configs/parte1.json`; produzir previsões internas `frame,id,x,y,w,h` e avaliá-las com o mesmo protocolo. O export MOT converte posições de volta a 1-based.
- **Parte 5:** o pipeline aceita detecções alternativas em CSV interno `frame,x,y,w,h,score`. Corromper as caixas públicas sem acesso ao GT; comparar baseline e modelo temporal nas mesmas entradas, sem retreinar.
- **Entrega:** este notebook explica a Parte 1. O `inferencia.ipynb` final será separado e usará o checkpoint da Pessoa 2. README oferece comandos por script; toda figura pode ser regenerada.
- Dados, pesos, caches, resultados volumosos e conversas permanecem fora do Git. O registro técnico de uso de IA exigido na entrega deve ser revisado sem incluir conversas ou informações pessoais.